# Market Impact Classification with Qwen-3-4B-Instruct

Classify GDELT events by whether they have a non-trivial impact on commodity markets (gold, silver, crude oil, natural gas, etc.)

In [2]:
!pip install -q transformers accelerate torch pandas

In [3]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

HF_token = user_secrets.get_secret("HF_TOKEN")
login(token=HF_token)

In [4]:
df = pd.read_csv("/kaggle/input/datasets/zygong1994/finance-world-model-dataset/gdelt_events_with_summaries.csv")
print(f"Total events: {len(df)}")
df.head(3)

Total events: 2394


,event_year,rank_in_year,GLOBALEVENTID,SQLDATE,Actor1Name,Actor1Type1Code,Actor1CountryCode,Actor2Name,Actor2Type1Code,Actor2CountryCode,...,ActionGeo_CountryCode,GoldsteinScale,NumArticles,AvgTone,impact_score,SOURCEURL,event_summary,key_actors,event_time,fetch_success
0,2015,1,435199599,20150522,IRAQI,GOV,IRQ,SHIA,UAF,NaN,...,IZ,-7.2,6,-53.962901,19.652643,http://freerepublic.com/focus/f-news/3292340/p...,Thousands of Shia militia fighters arrived at ...,"['IRAQI GOVERNMENT', 'SHIA MILITIAS', 'ISIS']",2015-05-22,False
1,2015,2,452406158,20150726,NIGERIA,NaN,NGA,NaN,NaN,NaN,...,NI,-10.0,20,-44.444444,18.246690,https://www.blesk.cz/clanek/zpravy-svet/332959...,A female suicide bomber detonated an explosive...,"['Nigeria', 'Boko Haram']",2015-07-26,False
2,2015,3,491126088,20151204,CZECH,NaN,CZE,NaN,NaN,NaN,...,EZ,-10.0,6,-36.363636,15.492864,http://www.blesk.cz/diskuse/488003,Czech Member of the European Parliament Milosl...,"['Miloslav Ransdorf', 'Swiss Police', 'Zürcher...",2015-12-04,False


In [23]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B-Instruct-2507")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-4B-Instruct-2507", device_map = 'auto')
print(f"model is loaded in {model.device}.")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model is loaded in cuda:0.


In [30]:
SYSTEM_PROMPT = (
    "You are a highly sensitive commodity market analyst. Given a news event summary, "
    "identify ANY potential for a non-trivial impact on commodity price markets. "
    "Err on the side of caution: if there is even a plausible ripple effect on supply, "
    "demand, or market sentiment—including indirect geopolitical or macroeconomic shifts—"
    "you MUST mark it as 1. "
    "Consider even minor disruptions in resource-rich zones, subtle trade policy hints, "
    "and speculative central bank shifts as impactful. "
    "Respond with ONLY the digit 1 (potential impact) or 0 (no conceivable impact). "
    "No explanation."
)


def build_prompt(event_summary: str) -> str:
    messages = [
        {"role": "user", "content": f"{SYSTEM_PROMPT}\n\nEvent: {event_summary}"},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


# Quick test
print(build_prompt(df["event_summary"].iloc[0]))

<|im_start|>user
You are a highly sensitive commodity market analyst. Given a news event summary, identify ANY potential for a non-trivial impact on commodity price markets. Err on the side of caution: if there is even a plausible ripple effect on supply, demand, or market sentiment—including indirect geopolitical or macroeconomic shifts—you MUST mark it as 1. Consider even minor disruptions in resource-rich zones, subtle trade policy hints, and speculative central bank shifts as impactful. Respond with ONLY the digit 1 (potential impact) or 0 (no conceivable impact). No explanation.

Event: Thousands of Shia militia fighters arrived at the Habbaniya military base in Anbar on May 22, 2015, to join Iraqi government forces in a planned counter-offensive against ISIS. The mobilization followed the fall of Ramadi and the capture of the nearby town of Husaybah by militants, signaling a significant escalation in the government's reliance on paramilitary forces to reclaim lost territory.<|im_

In [17]:
def classify_batch(summaries: list[str]) -> list[int]:
    """Classify a batch of event summaries. Returns list of 0/1."""
    prompts = [build_prompt(s) for s in summaries]
    inputs = tokenizer(
        prompts,
        add_generation_prompt=True,
    	tokenize=True,
    	return_dict=True,
    	return_tensors="pt",
        padding = True,
        padding_side='left',
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=2
        )

    results = []
    for i, output in enumerate(outputs):
        input_len = inputs["input_ids"][i].shape[0]
        generated = tokenizer.decode(output[input_len:], skip_special_tokens=True).strip()
        results.append(1 if generated.startswith("1") else 0)

    return results

In [24]:
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)
outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Hello! I'm Qwen, a large-scale language model independently developed by the Tongyi Lab under Alibaba Group. I'm proficient in 100 languages and can assist you with various tasks such


In [ ]:
BATCH_SIZE = 8

summaries = df["event_summary"].fillna("").tolist()
all_labels = []

for start in range(0, len(summaries), BATCH_SIZE):
    batch = summaries[start : start + BATCH_SIZE]
    labels = classify_batch(batch)
    all_labels.extend(labels)

    if (start // BATCH_SIZE) % 10 == 0:
        print(f"Processed {start + len(batch)}/{len(summaries)} events. \nProgress: {len(all_labels) / len(summaries):.2%}")
print(f"Done. Total classified: {len(all_labels)}")

In [ ]:
df["market_impact"] = all_labels

print(f"Market impact distribution:\n{df['market_impact'].value_counts()}")
print(f"\nImpact rate: {df['market_impact'].mean():.2%}")
df[["event_summary", "market_impact"]].head(10)

In [ ]:
df.to_csv("gdelt_events_with_market_impact.csv", index=False)
print("Saved to gdelt_events_with_market_impact.csv")